In [36]:
gen_report = False

In [37]:
import os
import json
import pandas as pd

root = "experiments"

records = []

for dirpath, dirnames, filenames in os.walk(root):
    if "report.json" in filenames:
        report_path = os.path.join(dirpath, "report.json")

        with open(report_path, "r") as f:
            data = json.load(f)

        record = {}

        record["VQEL"] = data.get("VQEL")

        # metrics
        for k, v in data.get("metrics", {}).items():
            record[k] = v

        # config
        for k, v in data.get("config", {}).items():
            if 'dir' in k.lower() or 'split' in k.lower():
                continue
            record[k] = v

        record["path"] = dirpath.replace('experiments/', '').replace('/results', '')
        records.append(record)

df = pd.DataFrame(records)

df["reset_unfrozen_params"] = df["reset_unfrozen_params"].map({True: "yes", False: "no"})
df["freeze_codebook"] = df["freeze_codebook"].map({True: "yes", False: "no"})
df["freeze_object_encoder"] = df["freeze_object_encoder"].map({True: "yes", False: "no"})

df.loc[df["num_iterations"] == 0, "learning_rate_tt"] = "-"
df.loc[df["agent_a_training_mode"] == 'frozen', "learning_rate_phase2_a"] = "-"
df.loc[df["best_of_n"].isna(), "best_of_n"] = 1
df.loc[df["dataset_tt"].isna(), "dataset_tt"] = "one_shape"
df.loc[df["sampling_temperature_tt"].isna(), "sampling_temperature_tt"] = df["sampling_temperature"]

df.loc[(df["num_iterations"] == 0) | (df["best_of_n"] == 1), "test_time_mode"] = "-"

df.rename(columns={"test_time_training_accuracy": "test_time_mutual_accuracy"}, inplace=True)
df.rename(columns={"test_time_self_play_accuracy": "test_time_self_accuracy"}, inplace=True)

df["test_time_mode"] = df["test_time_mode"].replace("adaptation", "sample_adaptation")

df = df.where(pd.notna(df), "None")
pd.options.display.float_format = '{:.10g}'.format
df = df.map(lambda x: f"{x:.0e}" if isinstance(x, (int, float)) and x != 0 and (abs(x) < 0.01 or abs(x) == 0.1 or abs(x) == 0.01) else x)
print("Total reports loaded:", len(df))


Total reports loaded: 243


/tmp/ipykernel_1436435/2064140999.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["num_iterations"] == 0, "learning_rate_tt"] = "-"
/tmp/ipykernel_1436435/2064140999.py:40: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["agent_a_training_mode"] == 'frozen', "learning_rate_phase2_a"] = "-"


In [38]:
def filter_df(filters, df=df, sort_by=["message_length", "message_length_tt", "learning_rate_tt", "num_iterations"]):

    idx = pd.Series(True, index=df.index)

    for key, values in filters.items():
        mask = pd.Series(False, index=df.index)
        if not isinstance(values, (list, tuple, set)):
            values = [values]
        for value in values:
            if value == "!None":
                mask |= (df[key] != "None")
            else:
                mask |= (df[key] == value)
        idx &= mask 

    return df[idx].sort_values(by=sort_by)
    

In [39]:
import base64

def to_html(df):    

    # ---- highlight rule ----
    if 'mutual_play_accuracy' in df:
        max_col = 'mutual_play_accuracy'
    else:
        max_col = 'test_accuracy'
    max_val = pd.to_numeric(df[max_col]).max()

    def highlight_max_row(row):
        if  pd.to_numeric(row[max_col]) == max_val:
            return ['font-weight: bold; background-color: #ffff99'] * len(row)
        else:
            return [''] * len(row)

    # ---- style ----
    styled = (
        df.style
            .apply(highlight_max_row, axis=1)  # <<< APPLY HIGHLIGHT HERE
            .hide(axis="index")
            .format(lambda x: f"{x:.0e}" if isinstance(x, (int, float)) and x != 0 and abs(x) < 0.01 else x)
            .set_table_styles([
                {"selector": "td", "props": [("border-right", "1px solid black")]},
                {"selector": "th", "props": [
                    ("border-right", "1px solid black"),
                    ("color", "darkblue"),
                    ("font-weight", "bold"),
                    ("padding-left", "8px"),
                    ("padding-right", "8px")
                ]},
            ])
            .set_properties(**{"text-align": "center"})
    )

    html_table = styled.to_html(index=False)

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_table)

        
def add_heading(num=1, title=""):
    """Add a professional-colored heading to results.html"""
    colors = ["#0b3d91",  # dark blue
              "#800000",  # maroon
              "#205522",  # dark green
              "#4b0082",  # indigo
              "#444444"]  # dark gray
    color = colors[(num-1) % len(colors)]  # cycle through colors
    
    if gen_report:
        with open("results.html", "a") as f:
            f.write(f'<h{num} style="color:{color}; font-family:Arial, sans-serif;">{title}</h{num}>\n')
        
def clear():
    if gen_report:
        with open("results.html", "w") as f:
            f.write("")

def line():
    if gen_report:
        with open("results.html", "a") as f:
            f.write('<hr style="border:1px solid #444; margin:10px 0;">\n')
        
        
def write(text):
    html_text = text.replace("\n", "<br>\n")
    
    if gen_report:
        with open("results.html", "a") as f:
            f.write(f"{html_text}\n")


def add_plot(path="plot.png"):
    """Embed an image file directly into results.html using base64."""
    with open(path, "rb") as img:
        encoded = base64.b64encode(img.read()).decode("utf-8")

    html_img = (
        '<img src="data:image/png;base64,' +
        encoded +
        '" style="max-width:650px; height:auto;"><br>\n'
    )

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_img)
        
    if os.path.exists(path):
        os.remove(path)

In [40]:
import os
import shutil

def remove_expr(res_df):
    for path in res_df["path"]:
        full_path = os.path.join("experiments", path)

        if not os.path.isdir(full_path):
            print(f"[SKIP] Not found: {full_path}")
            continue

        answer = input(f"⚠️ Delete expr '{path}' ? (y/n): ").strip().lower()

        if answer == "y":
            shutil.rmtree(full_path)
            print(f"✅ Deleted: {path}")
        else:
            print(f"❌ Skipped: {path}")

In [41]:
from itertools import product
import pandas as pd


def extract_maxes(df, cols=["seed", "agent_a_training_mode"], max_col="test_time_mutual_accuracy"):
    values = []
    for col in cols:
        value = set(df[col])
        values.append(value)

    combinations = [list(x) for x in product(*values)]

    maxes = []
    for comb in combinations:
        section = filter_df({col: value for col, value in zip(cols, comb)}, df)
        if not section.empty:
            max_row = section.loc[pd.to_numeric(section[max_col]).idxmax()]
            maxes.append(max_row.to_frame().T)

    final_df = pd.concat(maxes, ignore_index=True)
    return final_df.sort_values(by=cols)


def mean_and_std(df, out_cols=["VQEL", "dataset", "sim", "agent_a_training_mode"]):
    rows = []

    for i in range(0, len(df), 3):
        chunk = df.iloc[i:i+3]

        mean = chunk["mutual_play_accuracy"].mean() * 100
        std = chunk["mutual_play_accuracy"].std() * 100

        rows.append({
            col: chunk[col].iloc[0] for col in out_cols} | {
            "mutual_play_accuracy": f"{mean:.1f} ± {std:.1f}"
        })

    out = pd.DataFrame(rows)
    return out.sort_values(by=out_cols)
    

In [42]:
clear()

---

In [43]:
add_heading(1, "EXP1: ")

write(
"""
"""
)

In [44]:
backbone_cols = [
    "VQEL",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "mutual_play_accuracy",
    "test_time_mutual_accuracy",
    "test_time_self_accuracy",
    "path",
]

baseline_cols = [
    "VQEL",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "test_time_mutual_accuracy",
    "test_time_self_accuracy",
    "path",
]

scaling_cols = [
    "VQEL",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "sampling_temperature_tt",
    "best_of_n",
    "test_time_mutual_accuracy",
    "test_time_self_accuracy",
    "path",
]

scaling_cols_report = [
    "agent_a_training_mode",
    "learning_rate_phase1",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "self_play_accuracy_a",
    "mutual_play_accuracy",


]

adapt_cols = [
    "VQEL",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "learning_rate_tt",
    "num_iterations",
    "test_time_mutual_accuracy",
    "test_time_self_accuracy",
    "path",
]

adapt_cols_report = [
    "agent_a_training_mode",
    "learning_rate_phase1",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
]

# Shape - Euclidean

## Backbone

In [45]:
add_heading(2, "ShapeWorld")
add_heading(3, "VQEL - Euclidean")

res = filter_df({
    "dataset": "shape",
    "sim": "euclidean",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length"])
# res[backbone_cols]

In [46]:
final = extract_maxes(res, cols=["agent_a_training_mode", "message_length"])
final[backbone_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,mutual_play_accuracy,test_time_mutual_accuracy,test_time_self_accuracy,path
3,True,shape,euclidean,"[1, 2, 3, 4]",5,one_shape,-,0.726,0.765,None,20251221_0002_bs32_vocab10_repr1024_lr1_0.001_...
2,True,shape,euclidean,"[2, 3, 4]",5,one_shape,-,0.789,0.746,None,20251221_0043_bs32_vocab10_repr1024_lr1_0.001_...
0,True,shape,euclidean,"[3, 4]",5,one_shape,-,0.798,0.82,None,20251221_0128_bs32_vocab10_repr1024_lr1_0.001_...
1,True,shape,euclidean,[4],5,one_shape,-,0.847,0.74,None,20251221_0216_bs32_vocab10_repr1024_lr1_0.001_...


## Baseline

In [47]:
add_heading(3, "")

res = filter_df({
    "dataset": "shape",
    "sim": "euclidean",
    "dataset_tt": "two_shape",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length"])

res[baseline_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,test_time_self_accuracy,path
206,True,shape,euclidean,"[1, 2, 3, 4]",5,two_shape,-,0.591,0.489,20251223_0952_bs32_vocab10_repr1024_lr1_0.001_...
221,True,shape,euclidean,"[2, 3, 4]",5,two_shape,-,0.591,0.53,20251223_0934_bs32_vocab10_repr1024_lr1_0.001_...
207,True,shape,euclidean,[4],5,two_shape,-,0.612,0.539,20251223_0935_bs32_vocab10_repr1024_lr1_0.001_...


## Scaling

In [48]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "euclidean",
    "VQEL": True,
    "test_time_mode": "scaling"
}, sort_by=["message_length", "sampling_temperature_tt", "best_of_n"])

# res[scaling_cols]

In [49]:
final = extract_maxes(res, cols=["message_length"])
final[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_mutual_accuracy,test_time_self_accuracy,path
2,True,shape,euclidean,"[1, 2, 3, 4]",5,two_shape,scaling,1e-02,30,0.592,0.495,20251223_0218_bs32_vocab10_repr1024_lr1_0.001_...
1,True,shape,euclidean,"[2, 3, 4]",5,two_shape,scaling,1e-01,20,0.592,0.545,20251223_0329_bs32_vocab10_repr1024_lr1_0.001_...
0,True,shape,euclidean,[4],5,two_shape,scaling,1e-02,30,0.613,0.54,20251223_0549_bs32_vocab10_repr1024_lr1_0.001_...


## Adaptation

In [50]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "euclidean",
    "VQEL": True,
    "test_time_mode": ["sample_adaptation", "batch_adaptation"]
}, sort_by=["test_time_mode", "message_length_tt", "learning_rate_tt", "num_iterations"])

to_html(res[scaling_cols_report])

# res[scaling_cols]

In [51]:
final = extract_maxes(res, cols=["message_length"])
final[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_mutual_accuracy,test_time_self_accuracy,path
2,True,shape,euclidean,"[1, 2, 3, 4]",5,two_shape,sample_adaptation,1e-01,30,0.591,0.492,20251223_0302_bs32_vocab10_repr1024_lr1_0.001_...
1,True,shape,euclidean,"[2, 3, 4]",5,two_shape,sample_adaptation,1e-02,30,0.59,0.537,20251223_0447_bs32_vocab10_repr1024_lr1_0.001_...
0,True,shape,euclidean,[4],5,two_shape,sample_adaptation,1e-01,30,0.614,0.534,20251223_0641_bs32_vocab10_repr1024_lr1_0.001_...


# Shape - Cosine

## Backbone

In [52]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length"])

to_html(res[scaling_cols_report])

# res[scaling_cols]

In [53]:
final = extract_maxes(res, cols=["message_length"])
final[baseline_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,test_time_self_accuracy,path
3,True,shape,cosine,"[1, 2, 3, 4]",5,one_shape,-,0.853,None,20251220_1854_bs32_vocab10_repr1024_lr1_0.0001...
2,True,shape,cosine,"[2, 3, 4]",5,one_shape,-,0.891,None,20251220_0820_bs32_vocab10_repr1024_lr1_0.0001...
0,True,shape,cosine,"[3, 4]",5,one_shape,-,0.891,None,20251220_0615_bs32_vocab10_repr1024_lr1_0.001_...
1,True,shape,cosine,[4],5,one_shape,-,0.862,None,20251220_1028_bs32_vocab10_repr1024_lr1_0.001_...


## Baseline

In [54]:
add_heading(3, "")

res = filter_df({
    "dataset": "shape",
    "sim": "cosine",
    "dataset_tt": "two_shape",
    "VQEL": True,
    "message_length": "[3, 4]",
    "test_time_mode": "-"
}, sort_by=["message_length_tt"])

to_html(res[scaling_cols_report])

res[baseline_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,test_time_self_accuracy,path
172,True,shape,cosine,"[3, 4]",4,two_shape,-,0.467,0.387,20251225_1828_bs32_vocab10_repr1024_lr1_0.0001...
241,True,shape,cosine,"[3, 4]",5,two_shape,-,0.473,0.414,20251223_0933_bs32_vocab10_repr1024_lr1_0.001_...
43,True,shape,cosine,"[3, 4]",6,two_shape,-,0.476,0.417,20251225_1829_bs32_vocab10_repr1024_lr1_0.0001...
112,True,shape,cosine,"[3, 4]",7,two_shape,-,0.448,0.414,20251225_1834_bs32_vocab10_repr1024_lr1_0.0001...
38,True,shape,cosine,"[3, 4]",8,two_shape,-,0.424,0.411,20251225_1830_bs32_vocab10_repr1024_lr1_0.0001...
235,True,shape,cosine,"[3, 4]",9,two_shape,-,0.409,0.406,20251225_1831_bs32_vocab10_repr1024_lr1_0.0001...


## Scaling

In [55]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "cosine",
    "VQEL": True,
    "message_length": "[3, 4]",
    "test_time_mode": "scaling"
}, sort_by=["message_length", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

# res[scaling_cols]

In [56]:
final = extract_maxes(res, cols=["message_length_tt"])
final[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_mutual_accuracy,test_time_self_accuracy,path
0,True,shape,cosine,"[3, 4]",5,two_shape,scaling,1e-01,10,0.494,0.437,20251223_0017_bs32_vocab10_repr1024_lr1_0.001_...


## Adaptation

In [57]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "cosine",
    "VQEL": True,
    "message_length": "[3, 4]",
    "test_time_mode": ["sample_adaptation", "batch_adaptation"],
}, sort_by=["test_time_mode", "message_length_tt", "learning_rate_tt", "num_iterations"])

to_html(res[scaling_cols_report])

# res[adapt_cols]

In [58]:
final = extract_maxes(res, cols=["test_time_mode", "message_length_tt"])
final[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_mutual_accuracy,test_time_self_accuracy,path
1,True,shape,cosine,"[3, 4]",4,two_shape,batch_adaptation,1e-04,200,0.479,0.469,20251225_1807_bs32_vocab10_repr1024_lr1_0.0001...
2,True,shape,cosine,"[3, 4]",5,two_shape,batch_adaptation,1e-04,200,0.521,0.509,20251225_1809_bs32_vocab10_repr1024_lr1_0.0001...
3,True,shape,cosine,"[3, 4]",6,two_shape,batch_adaptation,1e-04,200,0.529,0.51,20251225_1812_bs32_vocab10_repr1024_lr1_0.0001...
4,True,shape,cosine,"[3, 4]",7,two_shape,batch_adaptation,1e-04,200,0.536,0.53,20251225_1814_bs32_vocab10_repr1024_lr1_0.0001...
5,True,shape,cosine,"[3, 4]",8,two_shape,batch_adaptation,1e-04,200,0.533,0.534,20251225_1817_bs32_vocab10_repr1024_lr1_0.0001...
6,True,shape,cosine,"[3, 4]",9,two_shape,batch_adaptation,1e-04,200,0.541,0.552,20251225_1819_bs32_vocab10_repr1024_lr1_0.0001...
7,True,shape,cosine,"[3, 4]",10,two_shape,batch_adaptation,1e-04,200,0.532,0.55,20251225_1822_bs32_vocab10_repr1024_lr1_0.0001...
0,True,shape,cosine,"[3, 4]",5,two_shape,sample_adaptation,1e-04,20,0.481,0.426,20251223_0123_bs32_vocab10_repr1024_lr1_0.001_...


# Shape12

## Backbone

In [59]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape12",
    "sim": "cosine",
    "VQEL": True,
    "dataset_tt": "shape12",
    "test_time_mode": "-"
}, sort_by=["message_length"])

to_html(res[scaling_cols_report])

res[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_mutual_accuracy,test_time_self_accuracy,path
100,True,shape12,cosine,"[3, 4]",10,shape12,-,1e-02,10,0.672,0.098,20251225_1907_bs32_vocab10_repr1024_lr1_0.0001...


## Baseline

In [60]:
add_heading(3, "")

res = filter_df({
    "dataset": "shape12",
    "sim": "cosine",
    "dataset_tt": "shape3",
    "VQEL": True,
    "message_length": "[3, 4]",
    "test_time_mode": "-"
}, sort_by=["message_length_tt"])

to_html(res[scaling_cols_report])

res[baseline_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,test_time_self_accuracy,path
91,True,shape12,cosine,"[3, 4]",4,shape3,-,0.664,0.59,20251225_2040_bs32_vocab10_repr1024_lr1_0.0001...
114,True,shape12,cosine,"[3, 4]",5,shape3,-,0.732,0.65,20251225_2050_bs32_vocab10_repr1024_lr1_0.0001...
171,True,shape12,cosine,"[3, 4]",6,shape3,-,0.737,0.616,20251225_2052_bs32_vocab10_repr1024_lr1_0.0001...
39,True,shape12,cosine,"[3, 4]",7,shape3,-,0.715,0.51,20251225_2054_bs32_vocab10_repr1024_lr1_0.0001...
55,True,shape12,cosine,"[3, 4]",8,shape3,-,0.658,0.336,20251225_2055_bs32_vocab10_repr1024_lr1_0.0001...
229,True,shape12,cosine,"[3, 4]",9,shape3,-,0.601,0.183,20251225_2056_bs32_vocab10_repr1024_lr1_0.0001...
197,True,shape12,cosine,"[3, 4]",10,shape3,-,0.569,0.087,20251225_2103_bs32_vocab10_repr1024_lr1_0.0001...


## Adaptation

In [61]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape12",
    "sim": "cosine",
    "VQEL": True,
    "message_length": "[3, 4]",
    "test_time_mode": ["sample_adaptation", "batch_adaptation"],
}, sort_by=["test_time_mode", "message_length_tt", "learning_rate_tt", "num_iterations"])

to_html(res[scaling_cols_report])

res[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_mutual_accuracy,test_time_self_accuracy,path
14,True,shape12,cosine,"[3, 4]",4,shape3,batch_adaptation,1e-04,200,0.507,0.567,20251225_1939_bs32_vocab10_repr1024_lr1_0.0001...
149,True,shape12,cosine,"[3, 4]",5,shape3,batch_adaptation,1e-04,200,0.616,0.674,20251225_1941_bs32_vocab10_repr1024_lr1_0.0001...
118,True,shape12,cosine,"[3, 4]",6,shape3,batch_adaptation,1e-04,200,0.713,0.778,20251225_1944_bs32_vocab10_repr1024_lr1_0.0001...
225,True,shape12,cosine,"[3, 4]",7,shape3,batch_adaptation,1e-04,200,0.706,0.773,20251225_1946_bs32_vocab10_repr1024_lr1_0.0001...
84,True,shape12,cosine,"[3, 4]",8,shape3,batch_adaptation,1e-04,200,0.595,0.775,20251225_1949_bs32_vocab10_repr1024_lr1_0.0001...
135,True,shape12,cosine,"[3, 4]",9,shape3,batch_adaptation,1e-04,200,0.445,0.801,20251225_1952_bs32_vocab10_repr1024_lr1_0.0001...
210,True,shape12,cosine,"[3, 4]",10,shape3,batch_adaptation,1e-04,200,0.378,0.811,20251225_1955_bs32_vocab10_repr1024_lr1_0.0001...


# MNIST

## Backbone

In [62]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "mnist1",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "-",
    "dataset_tt": "mnist1"
}, sort_by=["message_length", "message_length_tt"])

res[backbone_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,mutual_play_accuracy,test_time_mutual_accuracy,test_time_self_accuracy,path
22,True,mnist1,cosine,"[1, 2, 3, 4]",5,mnist1,-,0.893,0.925,0.848,20251225_0503_bs32_vocab10_repr192_lr1_0.0001_...
230,True,mnist1,cosine,"[2, 3, 4]",5,mnist1,-,0.911,0.944,0.845,20251225_0543_bs32_vocab10_repr192_lr1_0.0001_...
46,True,mnist1,cosine,"[3, 4]",2,mnist1,-,0.895,0.443,0.347,20251223_1750_bs32_vocab10_repr192_lr1_0.0001_...
17,True,mnist1,cosine,[4],5,mnist1,-,0.908,0.676,0.647,20251225_0626_bs32_vocab10_repr192_lr1_0.0001_...


## Baseline

In [63]:
add_heading(3, "")

res = filter_df({
    "dataset": "mnist1",
    "sim": "cosine",
    "dataset_tt": "mnist2",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length", "message_length_tt"])

res[baseline_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_mutual_accuracy,test_time_self_accuracy,path
59,True,mnist1,cosine,"[3, 4]",4,mnist2,-,0.439,0.464,20251224_1857_bs32_vocab10_repr192_lr1_0.0001_...
45,True,mnist1,cosine,"[3, 4]",5,mnist2,-,0.481,0.509,20251225_1756_bs32_vocab10_repr192_lr1_0.0001_...
19,True,mnist1,cosine,"[3, 4]",6,mnist2,-,0.5,0.534,20251223_1959_bs32_vocab10_repr192_lr1_0.0001_...
220,True,mnist1,cosine,"[3, 4]",7,mnist2,-,0.503,0.527,20251225_1757_bs32_vocab10_repr192_lr1_0.0001_...
34,True,mnist1,cosine,"[3, 4]",8,mnist2,-,0.495,0.518,20251224_0942_bs32_vocab10_repr192_lr1_0.0001_...
97,True,mnist1,cosine,"[3, 4]",9,mnist2,-,0.475,0.49,20251225_1758_bs32_vocab10_repr192_lr1_0.0001_...
62,True,mnist1,cosine,"[3, 4]",10,mnist2,-,0.436,0.46,20251224_1856_bs32_vocab10_repr192_lr1_0.0001_...


## Scaling

In [64]:
res = filter_df({
    "dataset": "mnist1",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "scaling"
}, sort_by=["message_length", "message_length_tt", "sampling_temperature_tt", "best_of_n"])

# res[scaling_cols]

In [65]:
final = extract_maxes(res, cols=["message_length", "message_length_tt"])
final[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_mutual_accuracy,test_time_self_accuracy,path
0,True,mnist1,cosine,"[3, 4]",4,mnist2,scaling,1e-01,100,1e-02,1e-02,20251224_1329_bs32_vocab1_repr192_lr1_0.0001_l...
1,True,mnist1,cosine,"[3, 4]",5,mnist2,scaling,1e-02,100,0.494,0.529,20251224_2337_bs32_vocab10_repr192_lr1_0.0001_...
2,True,mnist1,cosine,"[3, 4]",6,mnist2,scaling,1e-02,60,0.522,0.551,20251223_2042_bs32_vocab10_repr192_lr1_0.0001_...
3,True,mnist1,cosine,"[3, 4]",8,mnist2,scaling,1e-02,100,0.524,0.545,20251223_2157_bs32_vocab10_repr192_lr1_0.0001_...
4,True,mnist1,cosine,"[3, 4]",10,mnist2,scaling,1e-02,100,0.474,0.511,20251224_1034_bs32_vocab10_repr192_lr1_0.0001_...
5,True,mnist1,cosine,"[3, 4]",12,mnist2,scaling,1e-02,100,0.397,0.474,20251225_0102_bs32_vocab10_repr192_lr1_0.0001_...
6,True,mnist1,cosine,"[3, 4]",14,mnist2,scaling,1e-02,100,0.328,0.449,20251225_0226_bs32_vocab10_repr192_lr1_0.0001_...


## Adaptation

In [66]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "mnist1",
    "VQEL": True,
    "test_time_mode": ["sample_adaptation", "batch_adaptation"]
}, sort_by=["test_time_mode", "message_length_tt", "learning_rate_tt", "num_iterations"])

to_html(res[scaling_cols_report])

res[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_mutual_accuracy,test_time_self_accuracy,path
70,True,mnist1,cosine,"[3, 4]",4,mnist2,batch_adaptation,1e-04,200,0.472,0.536,20251225_1528_bs32_vocab10_repr192_lr1_0.0001_...
103,True,mnist1,cosine,"[3, 4]",5,mnist2,batch_adaptation,1e-04,200,0.54,0.605,20251225_1530_bs32_vocab10_repr192_lr1_0.0001_...
71,True,mnist1,cosine,"[3, 4]",6,mnist2,batch_adaptation,1e-04,200,0.589,0.664,20251225_1531_bs32_vocab10_repr192_lr1_0.0001_...
219,True,mnist1,cosine,"[3, 4]",7,mnist2,batch_adaptation,1e-04,200,0.604,0.686,20251225_1533_bs32_vocab10_repr192_lr1_0.0001_...
41,True,mnist1,cosine,"[3, 4]",8,mnist2,batch_adaptation,1e-04,200,0.617,0.705,20251225_1535_bs32_vocab10_repr192_lr1_0.0001_...
21,True,mnist1,cosine,"[3, 4]",9,mnist2,batch_adaptation,1e-04,200,0.62,0.719,20251225_1538_bs32_vocab10_repr192_lr1_0.0001_...
104,True,mnist1,cosine,"[3, 4]",10,mnist2,batch_adaptation,1e-04,200,0.609,0.731,20251225_1540_bs32_vocab10_repr192_lr1_0.0001_...
86,True,mnist1,cosine,"[3, 4]",4,mnist2,sample_adaptation,1e-04,5,0.444,0.475,20251224_1440_bs32_vocab10_repr192_lr1_0.0001_...
75,True,mnist1,cosine,"[3, 4]",4,mnist2,sample_adaptation,1e-04,10,0.419,0.456,20251224_1444_bs32_vocab10_repr192_lr1_0.0001_...
176,True,mnist1,cosine,"[3, 4]",4,mnist2,sample_adaptation,1e-04,15,0.386,0.427,20251224_1450_bs32_vocab10_repr192_lr1_0.0001_...


In [67]:
final = extract_maxes(res, cols=["test_time_mode", "message_length_tt"])
final[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_mutual_accuracy,test_time_self_accuracy,path
7,True,mnist1,cosine,"[3, 4]",4,mnist2,batch_adaptation,1e-04,200,0.472,0.536,20251225_1528_bs32_vocab10_repr192_lr1_0.0001_...
8,True,mnist1,cosine,"[3, 4]",5,mnist2,batch_adaptation,1e-04,200,0.54,0.605,20251225_1530_bs32_vocab10_repr192_lr1_0.0001_...
9,True,mnist1,cosine,"[3, 4]",6,mnist2,batch_adaptation,1e-04,200,0.589,0.664,20251225_1531_bs32_vocab10_repr192_lr1_0.0001_...
10,True,mnist1,cosine,"[3, 4]",7,mnist2,batch_adaptation,1e-04,200,0.604,0.686,20251225_1533_bs32_vocab10_repr192_lr1_0.0001_...
11,True,mnist1,cosine,"[3, 4]",8,mnist2,batch_adaptation,1e-04,200,0.617,0.705,20251225_1535_bs32_vocab10_repr192_lr1_0.0001_...
12,True,mnist1,cosine,"[3, 4]",9,mnist2,batch_adaptation,1e-04,200,0.62,0.719,20251225_1538_bs32_vocab10_repr192_lr1_0.0001_...
13,True,mnist1,cosine,"[3, 4]",10,mnist2,batch_adaptation,1e-04,200,0.609,0.731,20251225_1540_bs32_vocab10_repr192_lr1_0.0001_...
0,True,mnist1,cosine,"[3, 4]",4,mnist2,sample_adaptation,1e-05,10,0.445,0.476,20251224_1542_bs32_vocab10_repr192_lr1_0.0001_...
1,True,mnist1,cosine,"[3, 4]",5,mnist2,sample_adaptation,1e-04,5,0.491,0.52,20251225_0402_bs32_vocab10_repr192_lr1_0.0001_...
2,True,mnist1,cosine,"[3, 4]",6,mnist2,sample_adaptation,1e-04,5,0.512,0.562,20251223_1844_bs32_vocab10_repr192_lr1_0.0001_...


# ImageNet

## Backbone

In [68]:
res = filter_df({
    "dataset": "imagenet",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "-",
    "dataset_tt": "imagenet"
}, sort_by=["message_length", "message_length_tt"])

res[backbone_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,mutual_play_accuracy,test_time_mutual_accuracy,test_time_self_accuracy,path
223,True,imagenet,cosine,"[3, 4]",5,imagenet,-,0.893,0.714,0.749,20251225_1135_bs32_vocab10_repr2048_lr1_0.0001...


## Baseline

In [69]:
res = filter_df({
    "dataset": "imagenet",
    "dataset_tt": "imagenet_same_class",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length", "message_length_tt"])

to_html(res[scaling_cols_report])

res[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_mutual_accuracy,test_time_self_accuracy,path
151,True,imagenet,cosine,"[3, 4]",4,imagenet_same_class,-,1e-02,10,0.429,0.437,20251225_1737_bs32_vocab10_repr2048_lr1_0.0001...
187,True,imagenet,cosine,"[3, 4]",5,imagenet_same_class,-,1e-02,10,0.416,0.441,20251225_1739_bs32_vocab10_repr2048_lr1_0.0001...
10,True,imagenet,cosine,"[3, 4]",6,imagenet_same_class,-,1e-02,10,0.262,0.352,20251225_1745_bs32_vocab10_repr2048_lr1_0.0001...
212,True,imagenet,cosine,"[3, 4]",7,imagenet_same_class,-,1e-02,10,0.116,0.221,20251225_1740_bs32_vocab10_repr2048_lr1_0.0001...
157,True,imagenet,cosine,"[3, 4]",8,imagenet_same_class,-,1e-02,10,0.098,0.101,20251225_1746_bs32_vocab10_repr2048_lr1_0.0001...
150,True,imagenet,cosine,"[3, 4]",9,imagenet_same_class,-,1e-02,10,0.091,0.043,20251225_1741_bs32_vocab10_repr2048_lr1_0.0001...
92,True,imagenet,cosine,"[3, 4]",10,imagenet_same_class,-,1e-02,10,0.092,0.025,20251225_1748_bs32_vocab10_repr2048_lr1_0.0001...


## Adaptation

In [70]:
res = filter_df({
    "dataset": "imagenet",
    "VQEL": True,
    "test_time_mode": ["sample_adaptation", "batch_adaptation"]
}, sort_by=["test_time_mode", "message_length_tt", "learning_rate_tt", "num_iterations"])

res[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_mutual_accuracy,test_time_self_accuracy,path
232,True,imagenet,cosine,"[3, 4]",4,imagenet_same_class,batch_adaptation,1e-04,200,0.579,0.667,20251225_1455_bs32_vocab10_repr2048_lr1_0.0001...
208,True,imagenet,cosine,"[3, 4]",5,imagenet_same_class,batch_adaptation,1e-04,200,0.623,0.718,20251225_1458_bs32_vocab10_repr2048_lr1_0.0001...
168,True,imagenet,cosine,"[3, 4]",6,imagenet_same_class,batch_adaptation,1e-04,200,0.6,0.765,20251225_1501_bs32_vocab10_repr2048_lr1_0.0001...
136,True,imagenet,cosine,"[3, 4]",7,imagenet_same_class,batch_adaptation,1e-04,200,0.537,0.752,20251225_1506_bs32_vocab10_repr2048_lr1_0.0001...
72,True,imagenet,cosine,"[3, 4]",8,imagenet_same_class,batch_adaptation,1e-04,200,0.521,0.721,20251225_1511_bs32_vocab10_repr2048_lr1_0.0001...
196,True,imagenet,cosine,"[3, 4]",9,imagenet_same_class,batch_adaptation,1e-04,200,0.268,0.665,20251225_1516_bs32_vocab10_repr2048_lr1_0.0001...
226,True,imagenet,cosine,"[3, 4]",10,imagenet_same_class,batch_adaptation,1e-04,200,0.171,0.591,20251225_1522_bs32_vocab10_repr2048_lr1_0.0001...
214,True,imagenet,cosine,"[3, 4]",4,imagenet_same_class,sample_adaptation,1e-04,200,0.232,0.266,20251225_2329_bs32_vocab10_repr2048_lr1_0.0001...
68,True,imagenet,cosine,"[3, 4]",5,imagenet_same_class,sample_adaptation,1e-04,200,0.281,0.311,20251226_0102_bs32_vocab10_repr2048_lr1_0.0001...
194,True,imagenet,cosine,"[3, 4]",6,imagenet_same_class,sample_adaptation,1e-04,200,0.262,0.288,20251226_0251_bs32_vocab10_repr2048_lr1_0.0001...
